# Quantum Time Series Analysis: Reports
Compatible with Qiskit 1.2.4+

### Author
- Jacob L. Cybulski, Enquanted

### Date
- Feb 2025: Started

### Aims
> *This script aims to compare results obtained from running different quantum time series models.*

In [1]:
import sys
sys.path.append('.')
sys.path

['/home/jacob/miniconda3/envs/qiskit-gpu/lib/python311.zip',
 '/home/jacob/miniconda3/envs/qiskit-gpu/lib/python3.11',
 '/home/jacob/miniconda3/envs/qiskit-gpu/lib/python3.11/lib-dynload',
 '',
 '/home/jacob/miniconda3/envs/qiskit-gpu/lib/python3.11/site-packages',
 '.']

In [2]:
import os
import numpy as np
import pylab
import time
import copy
import pandas as pd
from tqdm.notebook import tqdm

from IPython.display import clear_output

from utils.Window import *
from utils.Charts import *
from utils.Files import *

import matplotlib.pyplot as plt
from matplotlib import set_loglevel
set_loglevel("error")
%matplotlib inline

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
### Listing control
debug = True
seed = 2022

### Software version
MAJOR = 9
MINOR = 12

### Constants
LOG_NAME = 'log_4_iter'
DATA_NAME = '2_sins'
DATA_PATH = f'{LOG_NAME}/data'
TRAIN_PATH = f'{LOG_NAME}/training'
ANALYSIS_PATH = f'{LOG_NAME}/analysis'
FIGURES_PATH = f'{LOG_NAME}/figures'
REPORTS_PATH = f'{LOG_NAME}/reports'

### Show constants
(LOG_NAME, DATA_NAME,DATA_PATH, TRAIN_PATH, ANALYSIS_PATH, FIGURES_PATH, REPORTS_PATH)

('log_4_iter',
 '2_sins',
 'log_4_iter/data',
 'log_4_iter/training',
 'log_4_iter/analysis',
 'log_4_iter/figures',
 'log_4_iter/reports')

## Utilities

In [4]:
def simple_report(df):

    # Extract the relevant info
    df.columns = df.columns.str.replace('_', '')
    df = df.replace({'_': '-'}, regex=True)
    report_data = df['Data'][0]
    report_model = df['Model'][0]
    report_epochs = df['Epochs'][0]
    report_label = f'tab:{report_data}-{report_model}'
    df = df.drop(['Data', 'Model', 'TrMedMAE', 'TsMedMAE'], axis=1)
    df = df[['Qubits', 'Params', 
             'TrMedR2', 'TrMedMSE', 'TsMedR2', 'TsMedMSE', 'Specs']]
    df.columns = ['Qubits', 'Params', 'TR2', 'TMSE', 'VR2', 'VMSE', 'Specs']
    vspace = '\\vspace{2mm}'

    report = df.style.hide().format(precision=4).to_latex(position_float='centering', 
        label=report_label, hrules=True, 
        column_format='rr@{\hskip 8pt}cc@{\hskip 8pt}cc@{\hskip 8pt}l',
        caption=f'Median model performance ({report_data}-{report_model}, ep={report_epochs}{vspace})')
    return report

In [5]:
def flex_report(df, position='t', caption=None, label=None, tab='    ', 
                precision=4, cformat=None, calias=None, hlines=[], add_epoch=False):
    
    # Extract the relevant info
    df.columns = df.columns.str.replace('_', '')
    df = df.replace({'_': '-'}, regex=True)
    data = df['Data'][0]
    model = df['Model'][0]
    epochs = df['Epochs'][0]
    df = df.drop(['Data', 'Model', 'TrMedMAE', 'TsMedMAE'], axis=1)
    df = df[['Qubits', 'Params', 
             'TrMedR2', 'TrMedMSE', 'TsMedR2', 'TsMedMSE', 'Specs']]
    cols = len(df.columns)
    rows = len(df)

    # Calculate defaults
    if cformat is None:
        cformat = "@{\\extracolsep{4pt}}rr@{\\hskip 4pt}rr@{\\hskip 4pt}rr@{\\hskip 4pt}l"

    if caption is None:
        caption = f'Median model performance ({model}/{data}'+(f', ep={epochs})' if add_epoch else ')')

    if label is None:
        label = f'tab:{data}-{model}'

    if calias is None:
        calias = ['Qubits', 'Params', 'R2', 'MSE', 'R2', 'MSE', 'Specs']

    # Generate report
    rp = ''

    rp += "\\begin{table}"+f"[{position}]\n"
    rp += "\\vspace{-5mm}\n"
    rp += "\\begin{center}\n"
    rp += "\\caption{"+f"{caption}"+"}\n"
    rp += "\\label{"+f"{label}"+"}\n"
    rp += "\\scriptsize\n"
    rp += "\\vspace{2mm}"
    rp += "\\begin{tabular}{ "+f"{cformat}"+" }\n"
    rp +=    f"{tab}"+"\\hline\n"
    rp +=    f"{tab}"+"& & \\multicolumn{2}{c}{Training} & \\multicolumn{2}{c}{Testing} & \\\\\n"
    rp +=    f"{tab}"+"\\cline{3-4}\\cline{5-6}\n"
    rp +=    f"{tab}"+" & ".join(calias)+'\\\\\n'
    rp +=    f"{tab}"+"\\hline\\hline\n"

    for row in range(rows):
        rp += f"{tab}"
        rp += f"{df['Qubits'][row]}"+" & "
        rp += f"{df['Params'][row]}"+" & "
        rp += f"{df['TrMedR2'][row]: 0.{precision}f}"+" & "
        rp += f"{df['TrMedMSE'][row]: 0.{precision}f}"+" & "
        rp += f"{df['TsMedR2'][row]: 0.{precision}f}"+" & "
        rp += f"{df['TsMedMSE'][row]: 0.{precision}f}"+" & "
        rp += "\\text{"+f"{df['Specs'][row]}"+"}"
        rp += '\\\\\n'
        if row in hlines:
            rp += f"{tab}"+"\\hdashline[2pt/1pt]\n"

    rp +=    f"{tab}"+"\\hline\n"
    rp += "\\end{tabular}\n"
    rp += "\\end{center}\n"
    rp += "\\vspace{-2mm}\n"
    rp += "\\end{table}\n\n"

    return rp

In [6]:
### Test
rf = '2_sins_parallel'
df = pd.read_csv(f'{REPORTS_PATH}/{rf}.tsv', delimiter='\t')
report = flex_report(df, hlines=[3, 7, 11])
print(report)

\begin{table}[t]
\vspace{-5mm}
\begin{center}
\caption{Median model performance (parallel/2-sins)}
\label{tab:2-sins-parallel}
\scriptsize
\vspace{2mm}\begin{tabular}{ @{\extracolsep{4pt}}rr@{\hskip 4pt}rr@{\hskip 4pt}rr@{\hskip 4pt}l }
    \hline
    & & \multicolumn{2}{c}{Training} & \multicolumn{2}{c}{Testing} & \\
    \cline{3-4}\cline{5-6}
    Qubits & Params & R2 & MSE & R2 & MSE & Specs\\
    \hline\hline
    2 & 48 &  0.0481 &  0.0456 & -0.1354 &  0.0354 & \text{q2 bl3 al3}\\
    3 & 72 &  0.0569 &  0.0452 & -0.1532 &  0.0359 & \text{q3 bl3 al3}\\
    4 & 96 & -0.0177 &  0.0488 & -0.2536 &  0.0391 & \text{q4 bl3 al3}\\
    5 & 120 &  0.8460 &  0.0074 &  0.6957 &  0.0095 & \text{q5 bl3 al3}\\
    \hdashline[2pt/1pt]
    5 & 60 &  0.7991 &  0.0096 &  0.7100 &  0.0090 & \text{q5 bl1 al1}\\
    5 & 120 &  0.8460 &  0.0074 &  0.6957 &  0.0095 & \text{q5 bl3 al3}\\
    5 & 180 &  0.7473 &  0.0121 &  0.6870 &  0.0098 & \text{q5 bl5 al5}\\
    5 & 240 &  0.8397 &  0.0077 &  0.6697 &  0

## Identify all required reports

In [7]:
### Define the required report names (as per log)
report_fname_list = ['2_sins_serial', '2_sins_parallel', '2_sins_xparallel',
                     '2_sins_sw_xqnn_ng', '2_sins_sw_ovload', '2_sins_sw_cnn']
report_mid_lines = [[], [3, 7, 11], [],
                    [4, 10], [], []]

## Produce reports for the selected models

In [8]:
### Generate reports for the selected models

print('\n\n')
for i in range(len(report_fname_list)):
    rf = report_fname_list[i]
    df = pd.read_csv(f'{REPORTS_PATH}/{rf}.tsv', delimiter='\t')
    report = flex_report(df, hlines=report_mid_lines[i], position='ht')
    print(report)




\begin{table}[ht]
\vspace{-5mm}
\begin{center}
\caption{Median model performance (serial/2-sins)}
\label{tab:2-sins-serial}
\scriptsize
\vspace{2mm}\begin{tabular}{ @{\extracolsep{4pt}}rr@{\hskip 4pt}rr@{\hskip 4pt}rr@{\hskip 4pt}l }
    \hline
    & & \multicolumn{2}{c}{Training} & \multicolumn{2}{c}{Testing} & \\
    \cline{3-4}\cline{5-6}
    Qubits & Params & R2 & MSE & R2 & MSE & Specs\\
    \hline\hline
    1 & 12 & -0.0495 &  0.0503 & -0.1320 &  0.0353 & \text{q1 l3}\\
    1 & 30 &  0.7768 &  0.0107 &  0.5166 &  0.0151 & \text{q1 l9}\\
    1 & 48 &  0.9616 &  0.0018 &  0.7514 &  0.0077 & \text{q1 l15}\\
    1 & 66 &  0.9966 &  0.0002 &  0.9192 &  0.0025 & \text{q1 l21}\\
    1 & 84 &  0.9977 &  0.0001 &  0.9107 &  0.0028 & \text{q1 l27}\\
    \hline
\end{tabular}
\end{center}
\vspace{-2mm}
\end{table}


\begin{table}[ht]
\vspace{-5mm}
\begin{center}
\caption{Median model performance (parallel/2-sins)}
\label{tab:2-sins-parallel}
\scriptsize
\vspace{2mm}\begin{tabular}{ @{\ext

## System

In [9]:
import os
import sys
print(f"\nOperating environment:\n")
os.system('lsb_release -d -s')
os.system('python --version')
print(f'Conda {os.path.basename(sys.prefix)}\n')


Operating environment:

Ubuntu 22.04.5 LTS
Python 3.11.11
Conda qiskit-gpu



In [10]:
print(f"\nSignificant Python packages:\n")
os.system('pip list | grep -e qiskit');


Significant Python packages:

qiskit                        1.2.4
qiskit-aer-gpu                0.15.1
qiskit-algorithms             0.3.1
qiskit-ibm-runtime            0.32.0
qiskit-machine-learning       0.8.1
qiskit-optimization           0.6.1
qiskit-sphinx-theme           1.16.1
